<a href="https://colab.research.google.com/github/nr868848-stack/E-commerce-Sales-Analysis/blob/main/Enterprise_Rag_Pipline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
langchain_community.vectorstores

NameError: name 'langchain_community' is not defined

In [ ]:
!pip install -q langchain langchain-community langchain-core sentence-transformers faiss-cpu pypdf rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os
from typing import List
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from sentence_transformers import CrossEncoder

class AdvancedRAGSystem:
    def __init__(self, embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        print("Initializing Embedding Engine & Vector Store Setup...")
        self.embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)
        # Advanced Re-Ranker for Precision Search
        self.reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        self.vector_store = None

    def process_and_index_pdf(self, pdf_path: str, chunk_size: int = 500, chunk_overlap: int = 100):
        """PDF Load, Chunk, Embedding creation, and Indexing Pipeline"""
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"File {pdf_path} not found.")

        print(f"Loading document: {pdf_path}...")
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()

        print("Applying Chunking strategy...")
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", " ", ""]
        )
        chunks = text_splitter.split_documents(documents)
        print(f"Total Chunks Created: {len(chunks)}")

        print("Building Vector Indexing in FAISS...")
        self.vector_store = FAISS.from_documents(chunks, self.embeddings)
        print("Indexing Complete Successfully!")

    def hybrid_retrieve_and_rerank(self, query: str, top_k: int = 10, final_top_n: int = 3) -> List[str]:
        """Retrieve initial matches and re-rank using Cross-Encoder"""
        if not self.vector_store:
            raise ValueError("Vector store is not initialized. Process a PDF first.")

        # Step 1: Initial Retrieval via Dense Embeddings
        initial_docs = self.vector_store.similarity_search(query, k=top_k)
        doc_texts = [doc.page_content for doc in initial_docs]

        # Step 2: Cross-Encoder Re-Ranking Pipeline
        pairs = [[query, text] for text in doc_texts]
        scores = self.reranker.predict(pairs)

        # Sort documents based on re-ranking scores
        scored_docs = sorted(zip(doc_texts, scores), key=lambda x: x[1], reverse=True)
        top_reranked_chunks = [doc for doc, score in scored_docs[:final_top_n]]

        return top_reranked_chunks

    def generate_prompt(self, query: str, context_chunks: List[str]) -> str:
        """Construct structured Grounded RAG Prompt"""
        context_str = "\n\n---\n\n".join(context_chunks)
        prompt = f"""
[SYSTEM INSTRUCTION]
You are an expert AI assistant. Answer the user's question strictly using only the provided context below.
If the answer cannot be determined from the context, state "Information not available in the source document."

[CONTEXT]
{context_str}

[USER QUESTION]
{query}

[ANSWER]
"""
        return prompt

# ==========================================
# EXECUTION PIPELINE
# ==========================================

# 1. Initialize RAG Engine
rag_pipeline = AdvancedRAGSystem()

# 2. Upload/Provide a PDF Path in Colab (Sample PDF path)
# Colab sidebar se koi bhi PDF upload karein aur uska path yahan likhein
sample_pdf_path = "/content/sample_data/README.md"  # Replace with your .pdf file path

# Note: Testing ke liye agar PDF na ho to textual fallback:
if not os.path.exists("test_doc.pdf"):
    from langchain_core.documents import Document
    print("Creating sample test index...")
    sample_texts = [
        Document(page_content="Data Science models like LLMs use Vector Databases for fast retrieval."),
        Document(page_content="Cross-Encoders re-rank documents by calculating full cross-attention between Query and Document."),
        Document(page_content="FAISS is an open-source library developed by Meta for efficient similarity search.")
    ]
    rag_pipeline.vector_store = FAISS.from_documents(sample_texts, rag_pipeline.embeddings)
else:
    rag_pipeline.process_and_index_pdf("test_doc.pdf")

# 3. Query & Rerank Execution
query = "How do Cross-Encoders help in retrieval?"
retrieved_chunks = rag_pipeline.hybrid_retrieve_and_rerank(query, top_k=3, final_top_n=2)

# 4. Generate Final LLM Prompt
final_prompt = rag_pipeline.generate_prompt(query, retrieved_chunks)

print("\n" + "="*50)
print("GENERATED PROMPT FOR LLM:")
print("="*50)
print(final_prompt)

Initializing Embedding Engine & Vector Store Setup...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Creating sample test index...

GENERATED PROMPT FOR LLM:

[SYSTEM INSTRUCTION]
You are an expert AI assistant. Answer the user's question strictly using only the provided context below. 
If the answer cannot be determined from the context, state "Information not available in the source document."

[CONTEXT]
Cross-Encoders re-rank documents by calculating full cross-attention between Query and Document.

---

Data Science models like LLMs use Vector Databases for fast retrieval.

[USER QUESTION]
How do Cross-Encoders help in retrieval?

[ANSWER]



In [ ]:
from transformers import pipeline

print("Loading LLM Model for Text Generation...")
# Light-weight LLM for Colab (Zero GPU cost)
generator = pipeline("text-generation", model="google/flan-t5-base")

# Generate response from our structured prompt
print("Generating Final Answer...")
response = generator(final_prompt, max_new_tokens=150)

print("\n" + "="*50)
print("FINAL RAG OUTPUT:")
print("="*50)
print(response[0]['generated_text'])

Loading LLM Model for Text Generation...


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM',

Generating Final Answer...


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



FINAL RAG OUTPUT:

[SYSTEM INSTRUCTION]
You are an expert AI assistant. Answer the user's question strictly using only the provided context below. 
If the answer cannot be determined from the context, state "Information not available in the source document."

[CONTEXT]
Cross-Encoders re-rank documents by calculating full cross-attention between Query and Document.

---

Data Science models like LLMs use Vector Databases for fast retrieval.

[USER QUESTION]
How do Cross-Encoders help in retrieval?

[ANSWER]

